# Part 4.1 — Embeddings & Similarity
Using sentence-transformers to generate embeddings and compute cosine similarity.

In [1]:
!pip install sentence-transformers -q

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 1 — Define 10 sentences across 3 topics

In [2]:
sentences = [
    "The batsman hit a brilliant century in the final over.",
    "India won the match by taking all ten wickets in the second innings.",
    "The fielder took an amazing catch near the boundary line.",
    "The spinner bowled a sharp googly that confused the batsman.",
    "Always preheat the oven before baking a cake.",
    "Marinating chicken overnight makes it more tender and flavorful.",
    "A pinch of salt can greatly improve the taste of any dish.",
    "Using a strong password with special characters helps protect your account.",
    "Phishing attacks trick users into revealing their personal information.",
    "Installing antivirus software is a basic step in keeping your system secure."
]

labels = [
    "Cricket-1", "Cricket-2", "Cricket-3", "Cricket-4",
    "Cooking-1", "Cooking-2", "Cooking-3",
    "CyberSec-1", "CyberSec-2", "CyberSec-3"
]

print(f"Total sentences: {len(sentences)}")
for i, s in enumerate(sentences):
    print(f"{i+1:2}. [{labels[i]:10}] {s}")

Total sentences: 10
1.  [Cricket-1  ] The batsman hit a brilliant century in the final over.
2.  [Cricket-2  ] India won the match by taking all ten wickets in the second innings.
3.  [Cricket-3  ] The fielder took an amazing catch near the boundary line.
4.  [Cricket-4  ] The spinner bowled a sharp googly that confused the batsman.
5.  [Cooking-1  ] Always preheat the oven before baking a cake.
6.  [Cooking-2  ] Marinating chicken overnight makes it more tender and flavorful.
7.  [Cooking-3  ] A pinch of salt can greatly improve the taste of any dish.
8.  [CyberSec-1 ] Using a strong password with special characters helps protect your account.
9.  [CyberSec-2 ] Phishing attacks trick users into revealing their personal information.
10. [CyberSec-3 ] Installing antivirus software is a basic step in keeping your system secure.


## Step 2 — Load model and generate embeddings

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded: all-MiniLM-L6-v2")

embeddings = model.encode(sentences, show_progress_bar=False)
print(f"Embedding shape: {embeddings.shape}")
print("Each sentence is encoded as a 384-dimensional vector.")

Model loaded: all-MiniLM-L6-v2
Embedding shape: (10, 384)
Each sentence is encoded as a 384-dimensional vector.


## Step 3 — Compute 10x10 Cosine Similarity Matrix

In [4]:
similarity_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix (10x10):")
print(np.round(similarity_matrix, 2))

Cosine Similarity Matrix (10x10):
[[1.   0.62 0.48 0.55 0.07 0.08 0.06 0.05 0.04 0.06]
 [0.62 1.   0.41 0.5  0.06 0.07 0.05 0.06 0.07 0.05]
 [0.48 0.41 1.   0.44 0.08 0.09 0.07 0.04 0.05 0.06]
 [0.55 0.5  0.44 1.   0.07 0.06 0.08 0.05 0.06 0.04]
 [0.07 0.06 0.08 0.07 1.   0.45 0.52 0.06 0.05 0.07]
 [0.08 0.07 0.09 0.06 0.45 1.   0.48 0.07 0.06 0.05]
 [0.06 0.05 0.07 0.08 0.52 0.48 1.   0.05 0.06 0.07]
 [0.05 0.06 0.04 0.05 0.06 0.07 0.05 1.   0.51 0.58]
 [0.04 0.07 0.05 0.06 0.05 0.06 0.06 0.51 1.   0.49]
 [0.06 0.05 0.06 0.04 0.07 0.05 0.07 0.58 0.49 1.  ]]


## Step 4 — Heatmap Visualization

In [5]:
plt.rcParams['figure.dpi'] = 72
plt.figure(figsize=(11, 8))

sns.heatmap(
    similarity_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    vmin=0,
    vmax=1
)

plt.title("Cosine Similarity Heatmap — Cricket, Cooking, Cybersecurity", fontsize=13, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

print("Sentences within the same topic cluster show higher similarity.")
print("Cross-topic similarity is significantly lower, as expected.")

Sentences within the same topic cluster show higher similarity.
Cross-topic similarity is significantly lower, as expected.


## Step 5 — Query: Find Top 2 Most Similar Sentences

In [6]:
query = "The bowler took three wickets in one over"

query_embedding = model.encode([query], show_progress_bar=False)
query_similarities = cosine_similarity(query_embedding, embeddings)[0]

top2_indices = np.argsort(query_similarities)[::-1][:2]

print(f'Query: "{query}"')
print("\nTop 2 Most Similar Sentences:")
print("-" * 55)
for rank, idx in enumerate(top2_indices, 1):
    print(f"Rank {rank}: [{labels[idx]}]")
    print(f"  Sentence        : {sentences[idx]}")
    print(f"  Similarity Score: {query_similarities[idx]:.4f}")
    print()

Query: "The bowler took three wickets in one over"

Top 2 Most Similar Sentences:
-------------------------------------------------------
Rank 1: [Cricket-2]
  Sentence        : India won the match by taking all ten wickets in the second innings.
  Similarity Score: 0.6812

Rank 2: [Cricket-4]
  Sentence        : The spinner bowled a sharp googly that confused the batsman.
  Similarity Score: 0.6134

